# Test QET dgl and pyg models

In [1]:
import warnings
warnings.simplefilter("ignore")

import torch
import numpy as np
import random
# from mp_api.client import MPRester

import matgl
from matgl.config import DEFAULT_ELEMENTS
from matgl.ext._pymatgen_dgl import Structure2Graph as DGLGraph
from matgl.ext._pymatgen_pyg import Structure2Graph as PyGGraph

from matgl.graph._data_dgl import MGLDataset as DGLDataset
from matgl.graph._data_pyg import MGLDataset as PyGDataset

from matgl.models._qet_dgl import QET as QET_DGL
# from matgl.models._qet_pyg_complete import QET as QET_PYG
from matgl.models._qet_pyg import QET as QET_PYG

In [2]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)

In [3]:
### load toy data
import pickle
import gzip

def load_cache(filename="mp_cache.pkl.gz"):
    with gzip.open(filename, "rb") as f:
        data = pickle.load(f)
    
    return data["structures"], data["labels"]

structures, labels = load_cache()

In [4]:
element_types = DEFAULT_ELEMENTS

dgl_converter = DGLGraph(element_types=element_types, cutoff=5.0)
pyg_converter = PyGGraph(element_types=element_types, cutoff=5.0)

# dgl_dataset = DGLDataset(structures=structures, converter=dgl_converter, labels=labels, include_ref_charge=True)
dgl_dataset = DGLDataset(structures=structures, converter=dgl_converter, labels=labels, include_ref_charge=False)
pyg_dataset = PyGDataset(structures=structures, converter=pyg_converter, labels=labels)

Warning! Loading graphs from processed cache at ./MGLDataset.


Processing...
Done!


In [5]:
len(dgl_dataset) == len(pyg_dataset)

True

In [6]:
def split_dataset_indices(dataset_size, train_ratio=0.8, val_ratio=0.1, seed=0):
    set_seed(seed)

    indices = torch.randperm(dataset_size)

    train_end = int(train_ratio * dataset_size)
    val_end = int((train_ratio + val_ratio) * dataset_size)

    train_idx = indices[:train_end]
    val_idx   = indices[train_end:val_end]
    test_idx  = indices[val_end:]

    return train_idx, val_idx, test_idx

In [7]:
train_idx, val_idx, test_idx = split_dataset_indices(len(dgl_dataset))

In [8]:
# train_idx = train_idx.tolist()
# val_idx   = val_idx.tolist()
# test_idx  = test_idx.tolist()

train_dgl = [dgl_dataset[i] for i in train_idx]
val_dgl = [dgl_dataset[i] for i in val_idx]
test_dgl = [dgl_dataset[i] for i in test_idx]


train_pyg = [pyg_dataset[i] for i in train_idx]
val_pyg = [pyg_dataset[i] for i in val_idx]
test_pyg = [pyg_dataset[i] for i in test_idx]

In [9]:
train_dgl[0]

(Graph(num_nodes=51, num_edges=1604,
       ndata_schemes={'q_ref': Scheme(shape=(), dtype=torch.float32), 'frac_coords': Scheme(shape=(3,), dtype=torch.float32), 'node_type': Scheme(shape=(), dtype=torch.int32)}
       edata_schemes={'bond_vec': Scheme(shape=(3,), dtype=torch.float64), 'bond_dist': Scheme(shape=(), dtype=torch.float64), 'pbc_offset': Scheme(shape=(3,), dtype=torch.float32)}),
 tensor([[[13.2197,  0.0000,  0.0000],
          [ 0.0000, 13.2197,  0.0000],
          [ 0.0000,  0.0000,  5.0879]]]),
 tensor([0., 0.]),
 {'energies': tensor(-426.5486),
  'forces': tensor([[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0.

In [10]:
train_pyg[0]

(Data(edge_index=[2, 1604], num_nodes=51, pbc_offset=[1604, 3], node_type=[51], frac_coords=[51, 3]),
 tensor([[[13.2197,  0.0000,  0.0000],
          [ 0.0000, 13.2197,  0.0000],
          [ 0.0000,  0.0000,  5.0879]]]),
 tensor([0., 0.]),
 {'energies': tensor(-426.5486),
  'forces': tensor([[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0

In [11]:
## loaders
from functools import partial

# from matgl.graph._data_dgl import MGLDataLoader as DGLLoader
# from matgl.graph._data_pyg import MGLDataLoader as PyGLoader

from dgl.dataloading import GraphDataLoader
from torch.utils.data import DataLoader

from matgl.graph._data_dgl import collate_fn_pes as collate_fn_pes_dgl
from matgl.graph._data_pyg import collate_fn_pes as collate_fn_pes_pyg


In [12]:
collate_dgl = partial(collate_fn_pes_dgl, include_stress=True)
collate_pyg = partial(collate_fn_pes_pyg, include_stress=True)

# train_loader_dgl, val_loader_dgl, test_loader_dgl = DGLLoader(train_data=train_dgl,
#                                                                 val_data=val_dgl,
#                                                                 test_data=test_dgl,
#                                                                 collate_fn=collate_dgl,
#                                                                 batch_size=2,
#                                                                 num_workers=0,)
# train_loader_pyg, val_loader_pyg, test_loader_pyg = PyGLoader(train_data=train_pyg,
#                                                                 val_data=val_pyg,
#                                                                 test_data=test_pyg,
#                                                                 collate_fn=collate_pyg,
#                                                                 batch_size=2,
#                                                                 num_workers=0,)



val_loader_dgl = GraphDataLoader(test_dgl, shuffle=False, collate_fn=collate_dgl, batch_size=1)
val_loader_pyg = DataLoader(test_pyg, shuffle=False, collate_fn=collate_pyg, batch_size=1)

In [13]:
## model init
set_seed(42)

model_dgl = QET_DGL(element_types=element_types, is_intensive=False, use_smooth=True, rbf_type="SphericalBessel")
model_pyg = QET_PYG(element_types=element_types, use_smooth=True, use_warp=False, rbf_type="SphericalBessel")
# model_pyg_compl = QET_PYG_compl(element_types=element_types, use_smooth=True, use_warp=False, rbf_type="SphericalBessel")


In [14]:
### fix model initial weights
state_dict_dgl = model_dgl.state_dict()
state_dict_pyg = model_pyg.state_dict()
# state_dict_pyg_sub = model_pyg_sub.state_dict()

## copy dgl to pyg
for k, v in state_dict_dgl.items():
    if k in state_dict_pyg and state_dict_dgl[k].shape == state_dict_pyg[k].shape:
        state_dict_pyg[k] = state_dict_dgl[k].clone()

model_pyg.load_state_dict(state_dict_pyg)


<All keys matched successfully>

In [15]:
model_pyg.state_dict()["chi_readout.layers.0.weight"]

tensor([[ 0.0881, -0.0360, -0.1200,  ..., -0.0864,  0.0475,  0.0973],
        [ 0.1065,  0.0272, -0.0783,  ..., -0.0245,  0.0920,  0.0984],
        [ 0.0658, -0.0038,  0.0673,  ..., -0.0986,  0.0911,  0.1020],
        ...,
        [-0.0223,  0.0028, -0.0810,  ..., -0.0306, -0.0524, -0.0692],
        [ 0.0323, -0.0018,  0.0626,  ..., -0.0214, -0.0437,  0.0195],
        [-0.0273, -0.0767,  0.0474,  ...,  0.0348, -0.0347, -0.0679]])

In [16]:
model_dgl.state_dict()["chi_readout.layers.0.weight"]

tensor([[ 0.0881, -0.0360, -0.1200,  ..., -0.0864,  0.0475,  0.0973],
        [ 0.1065,  0.0272, -0.0783,  ..., -0.0245,  0.0920,  0.0984],
        [ 0.0658, -0.0038,  0.0673,  ..., -0.0986,  0.0911,  0.1020],
        ...,
        [-0.0223,  0.0028, -0.0810,  ..., -0.0306, -0.0524, -0.0692],
        [ 0.0323, -0.0018,  0.0626,  ..., -0.0214, -0.0437,  0.0195],
        [-0.0273, -0.0767,  0.0474,  ...,  0.0348, -0.0347, -0.0679]])

In [17]:
from matgl.apps._pes_pyg import Potential as Potential_PYG_
from matgl.apps._pes_dgl import Potential as Potential_DGL_


potential_dgl = Potential_DGL_(model=model_dgl)
potential_pyg = Potential_PYG_(model=model_pyg)


potential_dgl.eval()
potential_pyg.eval()


Potential(
  (model): QET(
    (bond_expansion): BondExpansion(
      (rbf): SphericalBesselFunction()
    )
    (activation): SiLU()
    (tensor_embedding): TensorEmbedding(
      (distance_proj1): Linear(in_features=3, out_features=64, bias=True)
      (distance_proj2): Linear(in_features=3, out_features=64, bias=True)
      (distance_proj3): Linear(in_features=3, out_features=64, bias=True)
      (emb): Embedding(89, 64)
      (emb2): Linear(in_features=128, out_features=64, bias=True)
      (act): SiLU()
      (linears_tensor): ModuleList(
        (0-2): 3 x Linear(in_features=64, out_features=64, bias=False)
      )
      (linears_scalar): ModuleList(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): Linear(in_features=128, out_features=192, bias=True)
      )
      (init_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (layers): ModuleList(
      (0-1): 2 x TensorNetInteraction(
        (linears_scalar): ModuleList(
          (0): 

### Check QET dgl version and pyg (subclass) version

In [18]:
for batch_dgl, batch_pyg in zip(val_loader_dgl, val_loader_pyg):
    print("==============")
    g_dgl, lat_dgl, state_dgl, e_dgl, f_dgl, s_dgl = batch_dgl
    g_pyg, lat_pyg, state_pyg, e_pyg, f_pyg, s_pyg = batch_pyg

    out_pyg = potential_pyg(g=g_pyg, lat=lat_pyg, state_attr=state_pyg)
    out_dgl = potential_dgl(g=g_dgl, lat=lat_dgl, state_attr=state_dgl)

    print("max diff energy:", torch.max(torch.abs(out_dgl[0] - out_pyg[0])))
    print("max diff force:", torch.max(torch.abs(out_dgl[1] - out_pyg[1])))
    print("max diff stress:", torch.max(torch.abs(out_dgl[2] - out_pyg[2])))
    # print("max diff charge:", torch.max(torch.abs(out_dgl[3] - out_pyg[3])))
    # assert torch.allclose(out_dgl[0], out_pyg[0], atol=1e-4)


max diff energy: tensor(6.5565e-07, grad_fn=<MaxBackward1>)
max diff force: tensor(1.0970e-06, grad_fn=<MaxBackward1>)
max diff stress: tensor(5.6294e-06, grad_fn=<MaxBackward1>)
max diff energy: tensor(0.0008, grad_fn=<MaxBackward1>)
max diff force: tensor(4.9287e-05, grad_fn=<MaxBackward1>)
max diff stress: tensor(0.0008, grad_fn=<MaxBackward1>)
max diff energy: tensor(0.0010, grad_fn=<MaxBackward1>)
max diff force: tensor(2.9320e-05, grad_fn=<MaxBackward1>)
max diff stress: tensor(0.0006, grad_fn=<MaxBackward1>)
max diff energy: tensor(0.0089, grad_fn=<MaxBackward1>)
max diff force: tensor(6.1324e-05, grad_fn=<MaxBackward1>)
max diff stress: tensor(0.0006, grad_fn=<MaxBackward1>)
max diff energy: tensor(0.0010, grad_fn=<MaxBackward1>)
max diff force: tensor(2.5525e-05, grad_fn=<MaxBackward1>)
max diff stress: tensor(0.0006, grad_fn=<MaxBackward1>)
max diff energy: tensor(0.0019, grad_fn=<MaxBackward1>)
max diff force: tensor(8.2479e-05, grad_fn=<MaxBackward1>)
max diff stress: tenso

In [19]:
g_dgl

Graph(num_nodes=48, num_edges=1224,
      ndata_schemes={'q_ref': Scheme(shape=(), dtype=torch.float32), 'frac_coords': Scheme(shape=(3,), dtype=torch.float32), 'node_type': Scheme(shape=(), dtype=torch.int32), 'pos': Scheme(shape=(3,), dtype=torch.float32), 'chi': Scheme(shape=(), dtype=torch.float32), 'hardness': Scheme(shape=(), dtype=torch.float32), 'sigma': Scheme(shape=(), dtype=torch.float32), 'hardness_inv': Scheme(shape=(), dtype=torch.float32), 'chi_hardness_inv': Scheme(shape=(), dtype=torch.float32), 'sum_q': Scheme(shape=(), dtype=torch.float32), 'sum_hardness_inv': Scheme(shape=(), dtype=torch.float32), 'sum_chi_hardness_inv': Scheme(shape=(), dtype=torch.float32), 'charge': Scheme(shape=(), dtype=torch.float32), 'elec_pot': Scheme(shape=(), dtype=torch.float32), 'node_feat': Scheme(shape=(66,), dtype=torch.float32), 'atomic_energy': Scheme(shape=(1,), dtype=torch.float32)}
      edata_schemes={'bond_vec': Scheme(shape=(3,), dtype=torch.float32), 'bond_dist': Scheme(shape

In [20]:
g_pyg

DataBatch(edge_index=[2, 1224], num_nodes=48, pbc_offset=[1224, 3], node_type=[48], frac_coords=[48, 3], batch=[48], ptr=[2], lattice=[1224, 3, 3], pbc_offshift=[1224, 3], pos=[48, 3])

In [21]:
len(out_pyg), len(out_dgl)

(4, 4)

In [22]:
# ## loss equi
# import torch.nn.functional as F

# loss_dgl = F.mse_loss(out_dgl, e_dgl)
# loss_pyg = F.mse_loss(out_pyg, e_pyg)

# print(loss_dgl.item(), loss_pyg.item())
# assert abs(loss_dgl.item() - loss_pyg.item()) < 1e-4